# 0. Preamble

In [ ]:
"""Spline-based function approximation model for machine learning.

This module provides a simple implementation of a piecewise linear spline model
for approximating one-dimensional functions. It serves as an educational example
of function approximation, demonstrating concepts like model complexity,
overfitting prevention through knot selection, and optimization.
"""

import numpy as np
from scipy.optimize import minimize
from functools import partial
from random import shuffle


class SplineModel:
    """A piecewise linear spline model for one-dimensional function approximation.

    This model fits a series of connected linear segments (splines) to data by
    optimizing both the placement and heights of knot points. The model starts
    with a grid of candidate splines and intelligently selects a subset based
    on data distribution and variance minimization.

    Parameters
    ----------
    range_x : tuple of float
        The (min, max) range of x values to model, e.g., (0, 10).
    num_splines : int
        The desired number of spline segments in the final model.
    splines_in_starting_grid : int
        The number of splines in the initial uniform grid before selection.

    Attributes
    ----------
    knots_x : list of float
        The x-coordinates of the knot points after fitting.
    knots_y : list of float
        The y-coordinates of the knot points after fitting.
    splines : callable
        The fitted piecewise function that can be called with x values.
    splines_in_grid : int
        The number of splines in the starting grid.

    Examples
    --------
    >>> model = SplineModel(range_x=(0, 10), num_splines=5,
    ...                     splines_in_starting_grid=20)
    >>> model.fit(X_train, y_train)
    >>> predictions = model.predict(X_test)
    """

    def __init__(self, range_x, num_splines, splines_in_starting_grid):
        self.range_x = range_x
        self.num_splines = num_splines
        self.splines_in_grid = splines_in_starting_grid

    def fit(self, X, y):
        """Fit the spline model to training data.

        The fitting process involves three main steps:
        1. Choose knot x-positions through grid creation, pruning, and selection
        2. Initialize knot y-values with a linear approximation
        3. Optimize knot y-values to minimize mean squared error

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Training input values.
        y : array-like of shape (n_samples,)
            Training target values.
        """
        self._initialize_knots_x()
        self._choose_knots_x(X, y)
        self._initialize_knots_y(X, y)
        self.optimize_knots_y(X, y)

        knots = list(zip(self.knots_x, self.knots_y))
        self.splines = self._build_splines(knots)

    def predict(self, X):
        """Predict target values for input data.

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Input values for which to predict targets.

        Returns
        -------
        y_pred : ndarray of shape (n_samples,)
            Predicted target values.
        """
        y_pred, membership = self.splines(X)
        return y_pred

    def spline_membership(self, X):
        """Determine which spline segment each input value belongs to.

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Input values to classify into spline segments.

        Returns
        -------
        membership : ndarray of shape (n_samples,)
            Integer array where each value indicates which spline segment
            (0 to num_splines-1) the corresponding input belongs to.
        """
        y_pred, membership = self.splines(X)
        return membership

    def _initialize_knots_x(self):
        """Select optimal x-positions for knot points.

        Creates an initial uniform grid of knots.
        """
        self.knots_x = list(np.linspace(self.range_x[0], self.range_x[1], self.splines_in_grid + 1))

    def _choose_knots_x(self, X, y):
        """Select optimal x-positions for knot points.

        Using an initial uniform grid of knots, prunes segments with
        insufficient data, then selects the best subset by minimizing
        variance within segments.

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Training input values.
        y : array-like of shape (n_samples,)
            Training target values.
        """

        knots_x = self.knots_x
        membership = self._get_spline_membership(X, knots_x)

        knots_x, membership = self._prune_splines(knots_x, membership)

        knots_x = self._select_knots_x(y, knots_x, membership, self.num_splines)

        self.knots_x = knots_x

    def _initialize_knots_y(self, X, y):
        """Initialize knot y-values using a linear approximation.

        Sets initial knot heights by fitting a line through the endpoints
        of the data and evaluating it at each knot x-position.

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Training input values.
        y : array-like of shape (n_samples,)
            Training target values.
        """
        slope = (y[np.argmax(X)] - y[np.argmin(X)]) / (np.max(X) - np.min(X))
        start_y = y[np.argmin(X)]
        knots_y = [start_y + slope * (x - self.range_x[0]) for x in self.knots_x]
        self.knots_y = knots_y

    def optimize_knots_y(self, X, y):
        """Optimize knot y-values to minimize mean squared error.

        Uses numerical optimization to find the best knot heights given
        fixed knot x-positions.

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Training input values.
        y : array-like of shape (n_samples,)
            Training target values.
        """
        results = minimize(
            self._eval_knot_heights, self.knots_y, (self.knots_x, X, y)
        )
        knots_y = list(results.x)
        self.knots_y = knots_y

    @staticmethod
    def _get_spline_membership(X, knots_x):
        """Assign each data point to a spline segment.

        Parameters
        ----------
        X : array-like of shape (n_samples,)
            Input values to assign to segments.
        knots_x : list of float
            The x-coordinates of knot points defining segments.

        Returns
        -------
        membership : list of list of int
            A list where membership[i] contains the indices of data points
            that fall within the i-th spline segment.
        """
        num_splines = len(knots_x) - 1
        membership = [[] for _ in range(num_splines)]
        for i, x in enumerate(X):
            for j, back_knot, forward_knot in zip(range(num_splines), knots_x[:-1], knots_x[1:]):
                if j == 0 and back_knot <= x <= forward_knot:
                    membership[j].append(i)
                    break
                elif j > 0 and back_knot < x <= forward_knot:
                    membership[j].append(i)
                    break
        return membership

    @staticmethod
    def _prune_splines(knots_x, membership):
        """Remove spline segments with insufficient data points.

        Iteratively removes knots that create segments with fewer than 2
        data points, merging adjacent segments until all have adequate data.

        Parameters
        ----------
        knots_x : list of float
            The x-coordinates of knot points.
        membership : list of list of int
            Current assignment of data points to segments.

        Returns
        -------
        knots_x : list of float
            Updated knot x-coordinates after pruning.
        membership : list of list of int
            Updated membership after merging under-populated segments.
        """
        knots_x = knots_x[:]
        num_splines = len(knots_x) - 1
        issue = True
        while issue:
            issue = False
            for i in range(1, num_splines):
                if len(membership[i]) < 2 or len(membership[i-1]) < 2:
                    issue = True
                    break
            if issue:
                knots_x.pop(i)
                num_splines -= 1
                membership = SplineModel._join_members(i, membership)
        return knots_x, membership

    @staticmethod
    def _select_knots_x(y, knots_x, membership, num_splines):
        """Select optimal subset of knots by minimizing within-segment variance.

        Iteratively removes knots that, when merged with adjacent segments,
        result in the smallest increase in total variance.

        Parameters
        ----------
        y : array-like of shape (n_samples,)
            Target values used to compute variance.
        knots_x : list of float
            Current knot x-coordinates.
        membership : list of list of int
            Current assignment of data points to segments.
        num_splines : int
            Desired number of final spline segments.

        Returns
        -------
        knots_x : list of float
            Selected knot x-coordinates.
        """
        splines_in_grid = len(membership)
        while len(membership) > num_splines:

            candidates = list(range(1, splines_in_grid))
            shuffle(candidates)

            best_variance = float('inf')
            for candidate in candidates:
                variance = SplineModel._mean_variance(y, SplineModel._join_members(candidate, membership))
                if variance < best_variance:
                    best_variance = variance
                    best_candidate = candidate

            knots_x.pop(best_candidate)
            splines_in_grid -= 1
            membership = SplineModel._join_members(best_candidate, membership)

        return knots_x

    @staticmethod
    def _mean_variance(y, membership):
        """Compute weighted mean variance across all segments.

        Parameters
        ----------
        y : array-like of shape (n_samples,)
            Target values.
        membership : list of list of int
            Assignment of data points to segments.

        Returns
        -------
        variance : float
            The weighted average of variance within each segment, weighted
            by the number of points in each segment.
        """
        counts = np.array([len(members) for members in membership])
        return (
            np.sum(
                counts * np.array([np.var(y[members]) for members in membership]))
            / np.sum(counts)
        )

    @staticmethod
    def _join_members(i, membership):
        """Merge two adjacent segments by removing their shared knot.

        Parameters
        ----------
        i : int
            Index of the knot to remove (1 to len(membership)-1).
        membership : list of list of int
            Current assignment of data points to segments.

        Returns
        -------
        membership : list of list of int
            Updated membership with segments i-1 and i merged.

        Raises
        ------
        AssertionError
            If i is 0 or len(membership), as these are endpoint knots.
        """
        assert 0 < i < len(membership), f'Cannot join on knot index {i} - this is an end knot'
        return (
            membership[:i-1] +
            [membership[i-1] + membership[i]] +
            membership[i+1:]
        )

    @staticmethod
    def _eval_knot_heights(knots_y, knots_x, X, y):
        """Evaluate mean squared error for given knot heights.

        Objective function for optimizing knot y-values.

        Parameters
        ----------
        knots_y : array-like
            Candidate y-coordinates for knots.
        knots_x : list of float
            Fixed x-coordinates of knots.
        X : array-like of shape (n_samples,)
            Training input values.
        y : array-like of shape (n_samples,)
            Training target values.

        Returns
        -------
        mse : float
            Mean squared error between predictions and targets.
        """
        knots = list(zip(knots_x, knots_y))
        splines = SplineModel._build_splines(knots)
        y_pred, _ = splines(X)
        return np.mean((y - y_pred) ** 2)

    @staticmethod
    def _range_condition(start, end, inclusive, x):
        """Create boolean array indicating if x values fall within a range.

        Parameters
        ----------
        start : float
            Start of the range.
        end : float
            End of the range.
        inclusive : bool
            If True, include start value; if False, exclude start value.
            End value is always included.
        x : array-like
            Values to test.

        Returns
        -------
        mask : ndarray of bool
            Boolean array indicating which x values fall in the range.
        """
        if not inclusive:
            return (start < x) & (x <= end)
        else:
            return (start <= x) & (x <= end)

    @staticmethod
    def _spline(start, end, x):
        """Evaluate a linear spline segment.

        Parameters
        ----------
        start : tuple of (float, float)
            The (x, y) coordinates of the start knot.
        end : tuple of (float, float)
            The (x, y) coordinates of the end knot.
        x : array-like
            Input values at which to evaluate the spline.

        Returns
        -------
        y : ndarray
            Output values of the linear spline at x.
        """
        slope = (end[1] - start[1]) / (end[0] - start[0])
        return start[1] + slope * (x - start[0])

    @staticmethod
    def _piecewise(conditions, funcs, x):
        """Evaluate a piecewise function and track segment membership.

        Parameters
        ----------
        conditions : list of callable
            Functions that return boolean arrays indicating which x values
            belong to each piece.
        funcs : list of callable
            Functions to evaluate for each piece.
        x : array-like
            Input values.

        Returns
        -------
        y : ndarray
            Output values from the piecewise function.
        spline_membership : ndarray of int
            Index of the spline segment each x value belongs to.
        """
        boolean_arrays = [condition(x) for condition in conditions]
        spline_membership = np.array(boolean_arrays).argmax(axis=0)
        return np.piecewise(
            x, boolean_arrays,
            funcs
        ), spline_membership

    @staticmethod
    def _build_splines(knots):
        """Construct a piecewise linear function from knot points.

        Parameters
        ----------
        knots : list of tuple of (float, float)
            The (x, y) coordinates of each knot point, in order.

        Returns
        -------
        spline_func : callable
            A function that takes x values and returns (y_values, membership).
        """
        last_knot = knots[0]
        funcs = []
        conditions = []
        for i, knot in enumerate(knots[1:]):
            funcs.append(partial(SplineModel._spline, last_knot, knot))
            conditions.append(
                partial(SplineModel._range_condition, last_knot[0], knot[0], i == 0)
            )
            last_knot = knot
        return partial(SplineModel._piecewise, conditions, funcs)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import explained_variance_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# 1. The Need for Function Approximators

There are many different relationships that data can take. Most of which are nonlinear.

In [ ]:
X = np.linspace(0, 10, 100)
ys = [
    0.5*X - 0.3,
    np.sin(X),
    2 / (1 + np.exp(-(X-5))),
    np.log(0.5 + X),
    np.sin(X)**3,
    np.abs(X-5)
]
dfs = []
for i, y in enumerate(ys):
    df = pd.DataFrame({'x': X, 'y': y})
    df['function'] = i
    dfs.append(df)
df = pd.concat(dfs)

px.line(df, x='x', y='y', color='function', facet_col='function', facet_col_wrap=3, height=700)

Discerning these patterns becomes harder as we add noise.

In [ ]:
X = np.linspace(0, 10, 100)
ys = [
    0.5*X - 0.3,
    np.sin(X),
    2 / (1 + np.exp(-(X-5))),
    np.log(0.5 + X),
    np.sin(X)**3,
    np.abs(X-5)
]
dfs = []
for i, y in enumerate(ys):
    df = pd.DataFrame({'x': X, 'y': y + np.random.random(y.shape) * 2})
    df['function'] = i
    dfs.append(df)
df = pd.concat(dfs)

px.scatter(df, x='x', y='y', color='function', facet_col='function', facet_col_wrap=3, height=700)

It'd be nice to be able to discover the signal, in the noise, without us having to guess at the relationship.

# 2. An Example of Fish Length at Age

In [ ]:
def von_bertalanffy(L_inf, k, t0, t):
    return L_inf * (1 - np.exp(-k * (t - t0)))

X = np.linspace(0, 10, 100)
y = von_bertalanffy(500, 2.5, 0, X)
px.line(x=X, y=y, labels={'x': 'Age', 'y': 'Length (cm)'})

# 3. The Spline Model

We'll use a silly model I've cooked up to do this function approximation.

In [ ]:
model = SplineModel(
    range_x=(0, 10),
    num_splines=25,
    splines_in_starting_grid=25
)
model._initialize_knots_x()
model._initialize_knots_y(X, y)

knots = list(zip(model.knots_x, model.knots_y))
splines = model._build_splines(knots)
y_pred, membership = splines(X)
px.line(
    x=np.concatenate([X, X]), y=np.concatenate([y_pred, y]),
    color=np.concatenate([membership, np.ones(y.shape) * (np.max(membership) + 1)]),
    labels={'x': 'Age', 'y': 'Length (cm)'}
)

In [ ]:
model.optimize_knots_y(X, y)

knots = list(zip(model.knots_x, model.knots_y))
splines = model._build_splines(knots)
y_pred, membership = splines(X)
px.line(
    x=np.concatenate([X, X]), y=np.concatenate([y_pred, y]),
    color=np.concatenate([membership, np.ones(y.shape) * (np.max(membership) + 1)]),
    labels={'x': 'Age', 'y': 'Length (cm)'}
)

We clearly don't need so many splines at the end.

In [ ]:
model = SplineModel(
    range_x=(0, 10),
    num_splines=10,
    splines_in_starting_grid=25
)
model._initialize_knots_x()
model._choose_knots_x(X, y)
model._initialize_knots_y(X, y)

knots = list(zip(model.knots_x, model.knots_y))
splines = model._build_splines(knots)
y_pred, membership = splines(X)
px.line(
    x=np.concatenate([X, X]), y=np.concatenate([y_pred, y]),
    color=np.concatenate([membership, np.ones(y.shape) * (np.max(membership) + 1)]),
    labels={'x': 'Age', 'y': 'Length (cm)'}
)

In [ ]:
model.optimize_knots_y(X, y)

knots = list(zip(model.knots_x, model.knots_y))
splines = model._build_splines(knots)
y_pred, membership = splines(X)
px.line(
    x=np.concatenate([X, X]), y=np.concatenate([y_pred, y]),
    color=np.concatenate([membership, np.ones(y.shape) * (np.max(membership) + 1)]),
    labels={'x': 'Age', 'y': 'Length (cm)'}
)

# 4. Putting it All Together - an API

Machine learning models usually hide all of the optimization so we don't need to worry about it and present instead a very simple `fit` and `predict` interface.

In [ ]:
model = SplineModel(
    range_x=(0, 10),
    num_splines=10,
    splines_in_starting_grid=25
)
model.fit(X, y)

y_pred = model.predict(X)
membership = model.spline_membership(X)
px.line(x=X, y=y_pred, color=membership, labels={'x': 'Age', 'y': 'Length (cm)'})

And we can try it across all of the other relationships we saw earlier.

In [ ]:
X = np.linspace(0, 10, 100)
ys = [
    0.5*X - 0.3,
    np.sin(X),
    2 / (1 + np.exp(-(X-5))),
    np.log(0.5 + X),
    np.sin(X)**3,
    np.abs(X-5)
]

NUM_SPLINES = 10

dfs = []
for i, y in enumerate(ys):
    df = pd.DataFrame({'x': X, 'y': y})
    df['function'] = i
    df['case'] = 'actual'
    dfs.append(df)

    model = SplineModel(
        range_x=(0, 10),
        num_splines=NUM_SPLINES,
        splines_in_starting_grid=25
    )
    model.fit(X, y)
    y_pred = model.predict(X)

    df = pd.DataFrame({'x': X, 'y': y_pred})
    df['function'] = i
    df['case'] = 'predicted'
    dfs.append(df)

df = pd.concat(dfs)

px.line(df, x='x', y='y', color='case', facet_col='function', facet_col_wrap=3, height=700)

# 5. Adding Noise

If we start adding noise some funky things start happening.

In [ ]:
np.random.seed(42)

def von_bertalanffy(L_inf, k, t0, t):
    return L_inf * (1 - np.exp(-k * (t - t0)))

X = np.linspace(0, 10, 100)
y = von_bertalanffy(500, 2.5, 0, X) + np.random.normal(0, 50, X.shape)
px.scatter(x=X, y=y, labels={'x': 'Age', 'y': 'Length (cm)'})

In [ ]:
model = SplineModel(
    range_x=(0, 10),
    num_splines=25,
    splines_in_starting_grid=25
)
model.fit(X, y)

y_pred = model.predict(X)
membership = model.spline_membership(X)
px.line(x=X, y=y_pred, color=membership, labels={'x': 'Age', 'y': 'Length (cm)'})

To prepare to see how to fix this, let's introduce a metric of how well our model has fit the data - explained variance. And then we'll change the number of splines and see how that changes the fit.

In [ ]:
rows = []
for num_splines in tqdm([5, 10, 15, 20, 25]):
    model = SplineModel(
        range_x=(0, 10),
        num_splines=num_splines,
        splines_in_starting_grid=50
    )
    model.fit(X, y)

    y_pred = model.predict(X)
    rows.append({
        'num_splines': num_splines,
        'explained_variance': explained_variance_score(y, y_pred)
    })

df = pd.DataFrame(rows)
px.line(df, x='num_splines', y='explained_variance')

This makes it look like there are no real limits to our performance, as we keep increasing `num_splines` the model just keeps improving! But we know as `num_splines` keeps increasing our fit of the actual signal is going down. So what do we do?

# 6. Hold Out Data!

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

In [ ]:
rows = []
for num_splines in tqdm([2, 3, 5, 10, 15, 20, 25]):
    model = SplineModel(
        range_x=(0, 10),
        num_splines=num_splines,
        splines_in_starting_grid=50
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_train)
    rows.append({
        'num_splines': num_splines,
        'explained_variance': explained_variance_score(y_train, y_pred),
        'case': 'train'
    })

    y_pred = model.predict(X_test)
    rows.append({
        'num_splines': num_splines,
        'explained_variance': explained_variance_score(y_test, y_pred),
        'case': 'test'
    })

df = pd.DataFrame(rows)
px.line(df, x='num_splines', y='explained_variance', color='case')

Our model is overfitting! 5 splines is the best score we get over the training data.

In [ ]:
model = SplineModel(
    range_x=(0, 10),
    num_splines=5,
    splines_in_starting_grid=50
)
model.fit(X_train, y_train)

y_pred = model.predict(X)
membership = model.spline_membership(X)
px.line(x=X, y=y_pred, color=membership, labels={'x': 'Age', 'y': 'Length (cm)'})

# 7. Adding More Data

Generally adding data allows us to use more complex models without overfitting.

In [ ]:
np.random.seed(42)

X_new = np.linspace(0, 10, 150)
y_new = von_bertalanffy(500, 2.5, 0, X_new) + np.random.normal(0, 50, X_new.shape)

X = np.concatenate([X, X_new])
y = np.concatenate([y, y_new])

y = y[np.argsort(X)]
X = X[np.argsort(X)]

px.scatter(x=X, y=y, labels={'x': 'Age', 'y': 'Length (cm)'})

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

rows = []
for num_splines in tqdm([2, 3, 5, 7, 10, 13, 15, 20]):
    model = SplineModel(
        range_x=(0, 10),
        num_splines=num_splines,
        splines_in_starting_grid=50
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rows.append({
        'num_splines': num_splines,
        'explained_variance': explained_variance_score(y_test, y_pred),
        'case': 'test'
    })

df = pd.DataFrame(rows)
px.line(df, x='num_splines', y='explained_variance', color='case')

In [ ]:
model = SplineModel(
    range_x=(0, 10),
    num_splines=7,
    splines_in_starting_grid=50
)
model.fit(X_train, y_train)

y_pred = model.predict(X)
membership = model.spline_membership(X)

y_perf = von_bertalanffy(500, 2.5, 0, X)

px.line(
    x=np.concatenate([X, X]), y=np.concatenate([y_pred, y_perf]),
    color=np.concatenate([membership, np.ones(y_perf.shape) * (np.max(membership) + 1)]),
    labels={'x': 'Age', 'y': 'Length (cm)'}
)

In [ ]:
print(
    "Training Explained Variance:",
    explained_variance_score(y_train, model.predict(X_train))
)
print(
    "Testing Explained Variance:",
    explained_variance_score(y_test, model.predict(X_test))
)

Moral of the story is that we combat noise with data.

# 8. Try it Yourself!

Here's another age - length relationship. See if you can fit a good spline model yourself.

In [ ]:
np.random.seed(42)

X = np.linspace(0, 10, 150)
y = von_bertalanffy(2000, 0.5, 0, X) + np.random.normal(0, 100, X.shape)

px.scatter(x=X, y=y, labels={'x': 'Age', 'y': 'Length (cm)'})

In [ ]:
# Code to Split Data

In [ ]:
# Code to Train Your Model

In [ ]:
# Code to Score Over Training

In [ ]:
# Code to Score Over Testing

In [ ]:
y_pred = model.predict(X)
membership = model.spline_membership(X)

y_perf = von_bertalanffy(2000, 0.5, 0, X)

px.line(
    x=np.concatenate([X, X]), y=np.concatenate([y_pred, y_perf]),
    color=np.concatenate([membership, np.ones(y_perf.shape) * (np.max(membership) + 1)]),
    labels={'x': 'Age', 'y': 'Length (cm)'}
)